In [11]:
import numpy as np
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/lafc_content.db')

In [12]:
with open('../sql/video_labels.sql') as f:
    query = f.read()

df = pd.read_sql_query(query, conn)
df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,primary_subject,n_playlists,all_playlists
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,Highlights,1.0,Highlights
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,Interviews,1.0,Interviews
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,NaN,NaN,NaN
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,NaN,NaN,NaN
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,Highlights,1.0,Highlights
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3643,8yoRN5MvJP0,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10267,164,16,0.01753,horizontal,NaN,NaN,NaN
3644,Emczin-vSFQ,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146333,1160,191,0.00923,horizontal,LAFC Essentials,2.0,LAFC Essentials | Best of LAFC
3645,a63ytKgBTAc,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2098,24,2,0.01239,horizontal,NaN,NaN,NaN
3646,QJP0ITdmAeo,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6297,72,5,0.01223,horizontal,NaN,NaN,NaN


In [13]:
df['duration_minutes'] = pd.to_timedelta(df['duration']).dt.total_seconds() / 60
df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,primary_subject,n_playlists,all_playlists,duration_minutes
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,Highlights,1.0,Highlights,0.866667
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,Interviews,1.0,Interviews,14.316667
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,NaN,NaN,NaN,0.250000
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,NaN,NaN,NaN,0.216667
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,Highlights,1.0,Highlights,0.216667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3643,8yoRN5MvJP0,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10267,164,16,0.01753,horizontal,NaN,NaN,NaN,2.000000
3644,Emczin-vSFQ,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146333,1160,191,0.00923,horizontal,LAFC Essentials,2.0,LAFC Essentials | Best of LAFC,2.000000
3645,a63ytKgBTAc,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2098,24,2,0.01239,horizontal,NaN,NaN,NaN,0.450000
3646,QJP0ITdmAeo,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6297,72,5,0.01223,horizontal,NaN,NaN,NaN,1.016667


In [14]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11650.22it/s]


In [15]:
PROTOTYPES = {
    'goal_clip': [
        # structured (with score line)
        "GOAL: M. Bogusz vs VAN, 1'",
        "Denis Bouanga breaks the tie! LAFC 2 - 1 HOU",
        "Diego Rossi opens the scoring, LAFC 1 - 0 Dallas",
        # short descriptive goal moments (no score line — the leaking shape)
        "Sonny picks his spot 🎯",
        "Near post finish by Bouanga 😮‍💨",
        "Bouanga chips the keeper | ALL ANGLES",
        "SONNY FROM DISTANCE 🚀",
        "Timmy's strike from the top of the box",
    ],

    'highlights': [
        "Highlights | LAFC vs FC Dallas",
        "Full Highlights | 3-0 | LAFC vs. Colorado Rapids",
        "MATCH HIGHLIGHTS | LAFC vs Seattle Sounders",
        "Every Goal From LAFC's Inaugural MLS Season",
        "11 Goals Over In Our Last 2 Games | Watch Them All",
    ],
    'interview': [
        "Cherundolo: We'll Need Effort Again Against Austin",
        "Ebobisse: Feeling more and more confident by the day",
        "Bradley Addresses Media After Mark-Anthony Kaye Trade",
        "Bouanga speaks on his hat trick",
        "Nguyen: This Is Where You Start To Play For Playoff Positions",
        "State Of The Union | Tom Penn",
        "Hollingshead: Huge Result For Us, Three Points On The Road",
    ],
    'feature': [
        "Get To Know Kwadwo Opoku",
        "LAFC Profile | From Norway to LA, Adama Diomande",
        "Building A Legacy | Carlos Vela's Past & Future With LAFC",
        "Join The Club | Juan Pinto",
        "The Call-Up | Christian Ramirez",
    ],
    'behind_the_scenes': [
        "Behind The Scenes | 2026 Primary Kit Shoot",
        "Sounds of Training | First Week Back",
        "A Look Behind The Scenes With Equipment Manager Scott Tranilla",
        "Inside the locker room after the win",
    ],
    'match_preview': [
        "LAFC at LA Galaxy - Match Preview",
        "Keys To The Match | LAFC vs Seattle",
        "Previewing the road trip to Colorado",
        "What to watch for ahead of LAFC vs Austin FC",
    ],
    'recap': [
        "2023 LAFC Season Recap",
        "Recap | LAFC vs Colorado Rapids",
        "Looking Back At The 2022 MLS Cup Run",
        "Year In Review | 2021 Season",
    ],
        'presser': [
        "Gareth Bale Introductory Press Conference",
        "Olivier Giroud - Introductory LAFC Press Conference",
        "Cengiz Ünder | First Press Conference | LAFC vs ATX",
        "Carlos Vela Returns Press Conference",
        "LAFC Media Availability",
        "Will Ferrell Groundbreaking Presser",
        "Mayor Eric Garcetti Groundbreaking Presser",
    ],

        'training': [
        "LAFC Kicks Off 2023 Preseason Training",
        "LAFC's First Day of Training at the Performance Center",
        "Training ahead of Champions Cup 💪",
        "Training Report Presented by BODYARMOR | Carlos Vela 6/28/22",
        "LAFC's First Training at Banc of California Stadium",
        "Squad mic'd up 🎤",
        "Son Heung-Min | First Training Session",
    ],

}

In [16]:
title_embeddings = model.encode(df['title'].fillna('').tolist(), show_progress_bar=False)

In [17]:
cat_names = list(PROTOTYPES.keys())
cat_vectors = []
for name in cat_names:
    ex_embs = model.encode(PROTOTYPES[name])   # embed that category's example titles
    cat_vectors.append(ex_embs.mean(axis=0))   # average -> one "centroid" per category
cat_vectors = np.vstack(cat_vectors)

In [18]:
from sentence_transformers import util

sims = util.cos_sim(title_embeddings, cat_vectors).numpy()
best_idx = sims.argmax(axis=1)
df['ml_label'] = [cat_names[i] for i in best_idx]
df['ml_score'] = sims.max(axis=1)
df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,primary_subject,n_playlists,all_playlists,duration_minutes,ml_label,ml_score
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,Highlights,1.0,Highlights,0.866667,feature,0.251857
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,Interviews,1.0,Interviews,14.316667,presser,0.459299
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,NaN,NaN,NaN,0.250000,goal_clip,0.455591
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,NaN,NaN,NaN,0.216667,goal_clip,0.395216
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,Highlights,1.0,Highlights,0.216667,goal_clip,0.488763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3643,8yoRN5MvJP0,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10267,164,16,0.01753,horizontal,NaN,NaN,NaN,2.000000,feature,0.455910
3644,Emczin-vSFQ,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146333,1160,191,0.00923,horizontal,LAFC Essentials,2.0,LAFC Essentials | Best of LAFC,2.000000,highlights,0.556917
3645,a63ytKgBTAc,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2098,24,2,0.01239,horizontal,NaN,NaN,NaN,0.450000,presser,0.405028
3646,QJP0ITdmAeo,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6297,72,5,0.01223,horizontal,NaN,NaN,NaN,1.016667,match_preview,0.554512


In [19]:
CHECK = ['Match Previews', 'Interviews', 'Highlights', 'Full Matches', 'Anatomy of a Goal']

sub = df[df['playlist'].isin(CHECK)]
ct = pd.crosstab(sub['playlist'], sub['ml_label'], normalize='index').round(2)
ct[[c for c in ct.columns if ct[c].max() >= 0.05]]     # hide near-empty columns

KeyError: 'playlist'

In [ ]:
CHECK = ['Match Previews', 'Interviews', 'Highlights', 'Full Matches', 'Anatomy of a Goal']
s = df[df['playlist'].isin(CHECK)].copy()

for threshold in np.arange(0.30, 0.60, 0.02):
    s['ct'] = np.where(s['ml_score'] >= threshold, s['ml_label'], None)
    lab = s[s['ct'].notna()]

    hits = (
        ((lab['playlist'] == 'Match Previews')    & (lab['ct'] == 'match_preview')) |
        ((lab['playlist'] == 'Interviews')        & lab['ct'].isin(['interview', 'presser'])) |
        ((lab['playlist'] == 'Highlights')        & lab['ct'].isin(['highlights', 'goal_clip'])) |
        ((lab['playlist'] == 'Full Matches')      & lab['ct'].isin(['highlights', 'recap'])) |
        ((lab['playlist'] == 'Anatomy of a Goal') & (lab['ct'] == 'goal_clip'))
    )

    print(f'{threshold:.2f}  precision {hits.mean():.1%}  '
          f'coverage {len(lab)/len(s):.1%}  (n={len(lab)})')

0.30  precision 76.2%  coverage 94.4%  (n=1107)
0.32  precision 76.7%  coverage 93.2%  (n=1093)
0.34  precision 77.4%  coverage 91.0%  (n=1068)
0.36  precision 78.0%  coverage 88.7%  (n=1040)
0.38  precision 78.3%  coverage 86.3%  (n=1012)
0.40  precision 79.7%  coverage 82.4%  (n=966)
0.42  precision 81.4%  coverage 77.7%  (n=911)
0.44  precision 83.3%  coverage 72.9%  (n=855)
0.46  precision 84.3%  coverage 67.7%  (n=794)
0.48  precision 85.7%  coverage 61.9%  (n=726)
0.50  precision 86.2%  coverage 55.8%  (n=654)
0.52  precision 86.8%  coverage 49.8%  (n=584)
0.54  precision 87.1%  coverage 44.2%  (n=518)
0.56  precision 88.2%  coverage 40.6%  (n=476)
0.58  precision 89.9%  coverage 36.2%  (n=425)


In [ ]:
df.to_sql('classified_videos', conn, if_exists='replace', index=False)

3648